# MUSANGA V3.1 · NB06 — PlanetScope & embedding spatial information

**Question:** Does fine-scale spatial context add meaningful and transferable information for estimating *Musanga* fraction beyond the established Sentinel-2 and satellite-embedding baseline?

**Stage 1 — exploration, not model fitting:** establish native resolution, available 4-band Planet annual/monthly imagery, annual 9-band NIR texture, 64-band 10 m embeddings and the existing 100 m UAV target cells. Display matched Planet and embedding patches, inspect 30/50/100 m supervision availability and spatial alignment. Compare native PlanetScope canopy texture with spatial variability in embeddings without confusing apparent training fit with generalization.

**Stage 2 — planned ablation, only after patch QA:** model comparison under identical targets and folds:
- **B0:** S2 annual + S2 quarterly + annual satellite embeddings (NB04B baseline).
- **B1:** B0 + handcrafted Planet texture (existing NB05 features).
- **B2:** B0 + multiscale **spatial texture of embeddings** (not merely their 64 central-pixel values).
- **B3:** B0 + Planet spatial CNN features.
- **B4:** B0 + embedding spatial CNN features.
- **B5:** B0 + both Planet and embedding CNN features.

First pilot at 50 m, then 30/100 m. **Same continuous UAV target**, same spatially held-out folds for within-site testing and leave-one-forest-out for geographic transfer. Hold out complete spatial areas/sites *before* patch extraction or learned representation training. Run supervised CNN embedding extraction out-of-fold, and fit all scalers/PCA/normalization from training folds only.

**Constraints:** Annual Planet image is 4.77 m and **may not be cloud-free**. Satellite embeddings are 10 m and already summarize S1/S2 information. Yoko Planet 2021 vs UAV 2020 is a known mismatch. Existing NB04B 100 m samples are used for preview only; no CNN trained in this first stage. No full-ROI 4.77 m imagery is assumed downloaded.


## 00 · Settings — exploratory preview only


In [ ]:
from pathlib import Path
FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                          Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
if FINAL is None:
    from google.colab import drive
    drive.mount('/content/drive')
    FINAL=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),
                              Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert FINAL and FINAL.is_dir(),'Musanga Final workspace missing'
ROOT=FINAL/'Outputs/V3_1'
NB05=ROOT/'05_sat_predictor_exploration'
NB04=ROOT/'04_satellite_predictors/input_layers'
NB04B=ROOT/'04B_roi_quarterly_LOSO'
OUT=ROOT/'06_spatial_planet_embeddings'
FIG=OUT/'figures'
for d in (OUT,FIG):d.mkdir(parents=True,exist_ok=True)
SITES=['Baego','MOS','Yangambi','Yoko']
PLANET_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2021}
EMBEDDING_YEAR={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
PREVIEW_PER_SITE=3
PREVIEW_PATCH_PIXELS=48      # ~229m Planet context for each selected 100m cell
EMB_PATCH_PIXELS=23          # ~230m at 10m embedding resolution
RANDOM_SEED=20261024
print('Workspace:',FINAL)


## 01 · Validate source products and review temporal/resolution limitations

Checks actual files, expected 19-band annual Planet, 9-band NIR texture and the existing 64-band annual embeddings. Annual Planet composition bands are:
1–4 screened mean B/G/R/N; 5–8 screened median; 9–12 raw mean; **13–16 raw median**; 17–19 raw valid count / screened valid count / number screened out.
The screened product is **heuristic only**; use the raw median as the stable first CNN baseline.

NB04B's original 100 m sample cache is *not recreated* or overwritten.


In [ ]:
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt
from rasterio.windows import Window
from rasterio.warp import transform as rio_transform
from rasterio.enums import Resampling
from pyproj import Transformer
from IPython.display import display

def planet_file(site,kind):
    root=NB05/('derived_features' if kind=='Texture' else 'input_layers')
    folder=f'Musanga_V31_NB05_UAV_ROI_{kind}'
    tag=f'NB05_UAV_ROI_{site}_{PLANET_YEAR[site]}_'
    pattern=tag+('Annual_19band*.tif' if kind=='Annual' else 'Texture_9band*.tif')
    matches=sorted((root/folder).glob(pattern))
    if len(matches)!=1:
        raise FileNotFoundError(f'{site} {kind}: expected one file, found {len(matches)}: {matches}')
    return matches[0]

def embed_file(site):
    return NB04/'EMB_ANNUAL'/f'{site}_EMB_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'

AUDIT=[]
for site in SITES:
    for family,filepath,expected in (
        ('PLANET_ANNUAL',planet_file(site,'Annual'),19),
        ('PLANET_TEXTURE',planet_file(site,'Texture'),9),
        ('EMB_ANNUAL',embed_file(site),64)):
        assert filepath.is_file(),filepath
        with rasterio.open(filepath) as src:
            assert src.count==expected,(site,family,'expected',expected,'found',src.count)
            assert src.crs and src.width>0 and src.height>0
            AUDIT.append(dict(site=site,family=family,year=PLANET_YEAR[site] if family!='EMB_ANNUAL' else EMBEDDING_YEAR[site],
                bands=src.count,pixel_size_m=round(abs(src.res[0]),3),
                width=src.width,height=src.height,crs=str(src.crs),
                size_MB=round(filepath.stat().st_size/1e6,2),path=str(filepath)))
AUDIT=pd.DataFrame(AUDIT)
display(AUDIT.drop(columns=['path']))
AUDIT.to_csv(OUT/'01_input_audit.csv',index=False)
assert (NB05/'07_final_input_audit/07_site_summary.csv').exists(),'Complete NB05 Section 07 audit first'
print('All 12 native products found: four forests × annual Planet/texture/embeddings.')
print('Note: The 10m embeddings are not 4.77m imagery; do not upscale them to claim new spatial detail.')


## 02 · Select existing 100 m UAV reference cells (identical NB04B targets)

Only select example cells; avoid random point selection from the full Planet image, which would include areas without UAV supervision. Sampling is stratified by low/medium/high Musanga abundance **within each site**, so each forest's previews show more than zero-cover background. Cell centers are computed from NB04B's exact S2-grid transform, not from row/col as if they were Planet raster indices.


In [ ]:
from affine import Affine
SAMPLE_PATH=NB04B/'samples/roi_cells_100m.parquet'
assert SAMPLE_PATH.is_file(),SAMPLE_PATH
DF=pd.read_parquet(SAMPLE_PATH)
assert {'site','row','col','mean_probability'}.issubset(DF.columns)
DF=DF.loc[DF.site.isin(SITES)].copy()
assert len(DF)>0
RNG=np.random.default_rng(RANDOM_SEED)
CENTERS={}
SELECTED=[]
for site in SITES:
    local=DF.loc[DF.site==site].copy()
    assert not local.empty,site
    # Existing 100m NB04B samples use the 10m S2 affine aggregated exactly 10×.
    raw=NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'
    assert raw.is_file(),raw
    with rasterio.open(raw) as ds:
        assert abs(ds.res[0]-10)<.05 and abs(ds.res[1]-10)<.05
        tfm=ds.transform*Affine.scale(10,10)
        crs=ds.crs
    for i in range(PREVIEW_PER_SITE):
        ordered=local.sort_values('mean_probability')
        idx=int(round((i+.5)*(len(ordered)-1)/PREVIEW_PER_SITE))
        sample=ordered.iloc[idx]
        cx,cy=tfm*(float(sample.col)+.5,float(sample.row)+.5)
        SELECTED.append(dict(site=site,group=str(sample.group),mosaic=str(sample.mosaic),
            fraction=float(sample.mean_probability),row=int(sample.row),col=int(sample.col),
            center_x=float(cx),center_y=float(cy),crs=str(crs)))
    CENTERS[site]=dict(transform=tfm,crs=crs)
PREVIEW=pd.DataFrame(SELECTED)
display(PREVIEW[['site','group','fraction','row','col']].round(3))
PREVIEW.to_csv(OUT/'02_preview_cells.csv',index=False)
print('Preview targets are UAV-model estimates at 100 m, NOT independent field labels.')


## 03 · Matched *native* Planet RGB/NIR and embedding-neighborhood previews

For the same UAV-target cell center, read a ~229 m **Planet raw annual median RGB patch** and a ~230 m **10 m satellite-embedding patch**. Embeddings contain 64 latent bands; visualize the first three PCA components as an exploratory false-colour composite (PCA fit only on the displayed patch, **not** for any prediction model). Preserve image pixel scales. This step helps spot spatial shifts, boundary masking, and structural patterns.

The common center is transformed to each raster's CRS if necessary. If a patch crosses a source footprint boundary it is padded with zeros and flagged by nonzero-band masks; no fabricated reflectance values.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
def native_patch(path,cx,cy,center_crs,size,bands):
    with rasterio.open(path) as ds:
        trans=Transformer.from_crs(center_crs,ds.crs,always_xy=True)
        x,y=trans.transform(cx,cy)
        cc,rr=~ds.transform*(x,y)
        start_col=int(np.floor(cc-size/2))
        start_row=int(np.floor(rr-size/2))
        w=Window(start_col,start_row,size,size)
        arr=ds.read(bands,window=w,boundless=True,fill_value=0).astype('float32')
        return arr,ds.crs,ds.res
def rgb_fixed(arr,valid):
    # Input arr is RGB band order; calculate one robust stretch per local patch.
    if np.count_nonzero(valid)<10:return np.ones((*valid.shape,3),dtype='float32')
    p=arr[:,valid]
    lo=np.percentile(p,2,axis=1)
    hi=np.percentile(p,98,axis=1)
    rgb=np.moveaxis(np.clip((arr-lo[:,None,None])/(hi[:,None,None]-lo[:,None,None]+1e-6),0,1),0,-1)
    rgb[~valid]=1
    return rgb
def embed_pca_rgb(cube):
    mask=np.isfinite(cube).all(axis=0)&np.any(cube!=0,axis=0)
    out=np.ones((*mask.shape,3),dtype='float32')
    if mask.sum()<10:return out,mask
    X=cube[:,mask].T
    X=StandardScaler().fit_transform(X)
    vals=PCA(n_components=3).fit_transform(X)
    pct_lo=np.percentile(vals,2,axis=0);pct_hi=np.percentile(vals,98,axis=0)
    out[mask]=np.clip((vals-pct_lo)/(pct_hi-pct_lo+1e-6),0,1)
    return out,mask

fig,axes=plt.subplots(len(PREVIEW),2,figsize=(8,3.0*len(PREVIEW)),layout='constrained')
rows=[]
for i,row in enumerate(PREVIEW.itertuples(index=False)):
    pa=planet_file(row.site,'Annual')
    eb=embed_file(row.site)
    # Planet annual 13-16 = B,G,R,N raw median; select RGB order 15,14,13
    p,_,pres=native_patch(pa,row.center_x,row.center_y,row.crs,PREVIEW_PATCH_PIXELS,[15,14,13])
    e,_,eres=native_patch(eb,row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    pmask=np.all(np.isfinite(p)&(p>0),axis=0)
    eviz,emask=embed_pca_rgb(e)
    axes[i,0].imshow(rgb_fixed(p,pmask));axes[i,1].imshow(eviz)
    axes[i,0].set_title(f'{row.site} | UAV {row.fraction*100:.1f}% | Planet RGB')
    axes[i,1].set_title('10 m embedding neighborhood (local PCA)')
    for ax in axes[i]:ax.axis('off')
    rows.append(dict(site=row.site,fraction=row.fraction,
                     planet_valid_pct=100*pmask.mean(),embedding_valid_pct=100*emask.mean(),
                     planet_pixel_size_m=abs(pres[0]),embedding_pixel_size_m=abs(eres[0])))
plt.savefig(FIG/'03_planet_vs_embedding_native_patches.png',dpi=140,bbox_inches='tight')
plt.show()
PATCH_QA=pd.DataFrame(rows)
display(PATCH_QA.round(2))
PATCH_QA.to_csv(OUT/'03_patch_coverage.csv',index=False)
print('IMPORTANT: local PCA colours are arbitrary across panels/sites and cannot be compared spectrally.')


## 04 · Spatial embeddings vs Planet: first quantitative exploratory features

To assess whether embeddings also contain spatial texture, summarize **within-patch heterogeneity** of their 64 channels. Compare each 100 m target cell's:
- Planet NIR local texture (from NB05's existing annual raster, sampled as local median);
- embedding spatial heterogeneity (median channelwise SD over a ~230 m context), plus embedding central-context mean differences.

**Descriptive only:** 12 deliberately stratified preview cells are too few for useful correlations or model fitting. This diagnostic checks that features are finite and nontrivial. Later we will extract these features across *all* labelled cells, use consistent physical windows (not just fixed pixel counts), and evaluate added value on geographic holdouts.


In [ ]:
from scipy.ndimage import median_filter
QA=[]
for row in PREVIEW.itertuples(index=False):
    e,_,_=native_patch(embed_file(row.site),row.center_x,row.center_y,row.crs,EMB_PATCH_PIXELS,list(range(1,65)))
    m=np.isfinite(e).all(axis=0)&np.any(e!=0,axis=0)
    if m.sum()<25:continue
    em=e[:,m]
    vari=np.std(em,axis=1)
    center=e[:,EMB_PATCH_PIXELS//2,EMB_PATCH_PIXELS//2]
    context=np.mean(em,axis=1)
    t,_,_=native_patch(planet_file(row.site,'Texture'),row.center_x,row.center_y,row.crs,
                       PREVIEW_PATCH_PIXELS,[1,2,3,4])
    tm=np.isfinite(t).all(axis=0)&(t>=0).all(axis=0)&np.any(t>0,axis=0)
    QA.append(dict(site=row.site,uav_fraction=row.fraction,
        embedding_channel_sd_median=float(np.median(vari)),
        embedding_context_difference=float(np.mean(abs(center-context))),
        planet_NIR_CV_r3_median=float(np.median(t[1,tm])) if tm.any() else np.nan,
        valid_embedding_pct=100*m.mean(),valid_planet_texture_pct=100*tm.mean()))
Q=pd.DataFrame(QA)
display(Q.round(4))
Q.to_csv(OUT/'04_spatial_feature_feasibility.csv',index=False)
fig,axes=plt.subplots(1,2,figsize=(10,4),layout='constrained')
for site,group in Q.groupby('site'):
    axes[0].scatter(group.uav_fraction*100,group.embedding_channel_sd_median,label=site)
    axes[1].scatter(group.uav_fraction*100,group.planet_NIR_CV_r3_median,label=site)
axes[0].set(xlabel='UAV Musanga fraction (%)',ylabel='Median channel SD in embedding patch',
            title='Embeddings: spatial heterogeneity')
axes[1].set(xlabel='UAV Musanga fraction (%)',ylabel='Planet NIR CV (radius 3)',
            title='Planet: native-scale texture')
axes[1].legend()
plt.savefig(FIG/'04_spatial_heterogeneity_feasibility.png',dpi=150,bbox_inches='tight')
plt.show()
print('This plot has only 12 stratified examples. Do NOT interpret it as prediction skill.')


## 05 · Decision gate and forthcoming modelling experiment

Before CNN model fitting, confirm:
1. Patch-validity percentages and visual alignment of Planet with its target cells; independent UAV orthomosaic/probability overlay will follow when locating the original teacher rasters.
2. Check that Planet **2021** in Yoko is an intentional temporal mismatch against UAV **2020**.
3. Use 50 m targets first and compare 30/100 m; target extraction must use the original **full-resolution UAV probability rasters**, avoiding interpolation of 100 m labels.
4. Train compact models with group/spatial folds: **100 m/50 m patches overlap heavily**, so randomly split cells is not an acceptable test.
5. Compare `B0 → B1/B2 → B3/B4 → B5` using the same site holdout and identical target cells; summarize MAE (pp), RMSE (pp), R², high-cover bias, calibration, and performance separately at MOS. Aggregate CNN output to target cells fairly.

**NB06 currently implements the exploratory entry point (Sections 00–04) and experimental specification; CNN training follows after visual QA.** Avoid building a large training architecture before verifying supervision and patch geometry.


## 06 · Verify original UAV probability rasters and Planet co-registration

The authoritative teacher rasters are the same **M2 warmstart full-mosaic U-Net probability GeoTIFFs** used by NB04 (not the 100 m aggregate). We inspect their CRS, footprint, NoData encoding, dynamic range, and cross-check the reconstructed 100 m values against NB04B sampled targets for each mosaic. The code automatically tests several plausible uint16 denominators but **refuses to accept a scale** unless the agreement is good. Do not retrain anything if this fails.

Generate site/mosaic **side-by-side Planet RGB, original UAV orthomosaic RGB, and continuous UAV probability** panels with one common geographic extent at native pixels. The images may have different sensor dates and radiometry. The purpose is to check obvious geometric shifts, not to infer identity from colours.

*Important*: UAV 2020 and Planet 2021 for Yoko; one-year mismatch is logged. A visually plausible image is not proof of subpixel registration; if the overlay looks shifted, fix georeferencing upstream before extracting labels.


In [ ]:
# 06 — Authoritative UAV teacher paths and raster metadata
import numpy as np,pandas as pd,rasterio,matplotlib.pyplot as plt, json,math
from rasterio.warp import reproject,Resampling,transform_bounds
from rasterio.windows import from_bounds,Window
from rasterio.features import rasterize
from affine import Affine
from shapely.geometry import box
from IPython.display import display

UAV_TEACHERS=FINAL/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
UAV_MOSAICS={
 'Baego':['Baego.tif'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif','Mos_5_6_transparent_mosaic_group1_georef.tif'],
 'Yangambi':['Yangambi_georef.tif'],
 'Yoko':['Yoko_kvo_georef.tif']
}
MANIFEST_FILE=FINAL/'Outputs/V3_1/00_input_audit/00_input_manifest.json'
assert MANIFEST_FILE.exists(),MANIFEST_FILE
uav_manifest=json.loads(MANIFEST_FILE.read_text())
MOSAIC_PATH={x['file']:Path(x['path']) for x in uav_manifest['mosaics']}
for site,names in UAV_MOSAICS.items():
    for name in names:
        assert name in MOSAIC_PATH and MOSAIC_PATH[name].is_file(), ('Missing orthomosaic',site,name)
def teacher_file(name):
    return UAV_TEACHERS/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
SOURCE_TABLE=[]
for site,names in UAV_MOSAICS.items():
    for name in names:
        path=teacher_file(name)
        assert path.is_file(),f'Authoritative UAV model output missing: {path}'
        with rasterio.open(path) as ds:
            assert ds.count>=1 and ds.crs and ds.width>0 and ds.height>0
            # Distributed window samples for a rapid encoding diagnostic.
            samples=[]
            for frac_y,frac_x in ((.25,.25),(.5,.5),(.75,.75),(.3,.7),(.7,.3)):
                win=Window(max(0,int(ds.width*frac_x)-64),max(0,int(ds.height*frac_y)-64),
                           min(128,ds.width),min(128,ds.height))
                samples.append(ds.read(1,window=win))
            z=np.concatenate([a.reshape(-1) for a in samples])
            SOURCE_TABLE.append(dict(site=site,mosaic=name,crs=str(ds.crs),
                width=ds.width,height=ds.height,res_x=ds.res[0],res_y=ds.res[1],
                dtype=ds.dtypes[0],nodata=ds.nodata,
                sample_min=float(np.min(z)),sample_p99=float(np.percentile(z,99)),
                sample_max=float(np.max(z)),file=str(path)))
display(pd.DataFrame(SOURCE_TABLE).drop(columns='file'))
pd.DataFrame(SOURCE_TABLE).to_csv(OUT/'06_uav_teacher_audit.csv',index=False)
print('Five original M2 warmstart probability maps and orthomosaics located. No labels modified.')


## 07 · Memory-safe UAV probability calibration

The original version exhausted Colab RAM by executing `src.read(1, masked=True)` on an entire high-resolution UAV probability mosaic and allocating several more full-sized arrays. This version uses **GDAL WarpedVRT streaming**: only the small destination satellite grid is materialized. The VRT alpha band supplies footprint validity; this is an approximate coverage estimate at partial edge pixels, not an independent cloud/landcover mask. Comparison against existing verified 100 m UAV targets remains mandatory. If the original teacher file has undeclared NoData, fix the source metadata before trusting edge cells; do not lower calibration thresholds to force success.

The same streaming function is reused automatically by Section 09 to generate 50 m labels. **Restart Colab**, rerun setup and Sections 06–10. No original datasets are modified.


In [ ]:
# 07 — Calibrate against the existing NB04B 100 m reference targets.
# Cells 00–02 are required (DF, NB04, CENTERS); rerun those after a fresh session.
UAV_SCALE={}
REFERENCE_DIAGNOSTICS=[]
SCALE_CANDIDATES=(1.,1000.,10000.,65534.,65535.)
CACHE_ROOT=OUT/'teacher_50m_cache';CACHE_ROOT.mkdir(exist_ok=True,parents=True)
def satellite_grid(site,scale):
    path=NB04/'RAW_ANNUAL'/f'{site}_RAW_ANNUAL_{EMBEDDING_YEAR[site]}_10m.tif'
    with rasterio.open(path) as ds:
        assert ds.crs and abs(ds.res[0]-10)<.05
        return dict(crs=ds.crs,transform=ds.transform*Affine.scale(scale/10,scale/10),
                    width=int(math.ceil(ds.width*10/scale)),
                    height=int(math.ceil(ds.height*10/scale)))
def warp_raw_teacher(name,site,scale):
    """Memory-safe reprojection: GDAL streams the UAV GeoTIFF from disk.

    Never call src.read() on the whole orthomosaic-scale probability raster.
    Output arrays are only as large as the satellite grid (~hundreds of pixels).
    A warped alpha band estimates fractional footprint coverage; direct GDAL
    averaging ignores source NoData when the dataset declares it.
    """
    from rasterio.vrt import WarpedVRT
    g=satellite_grid(site,scale)
    with rasterio.open(teacher_file(name)) as src:
        assert src.count>=1 and src.crs
        vrt_opts=dict(crs=g['crs'],transform=g['transform'],
                      width=g['width'],height=g['height'],
                      resampling=Resampling.average,add_alpha=True)
        with WarpedVRT(src,**vrt_opts) as vrt:
            # First channel is the source teacher; added alpha is the last.
            value=vrt.read(1,masked=True,out_dtype='float32')
            alpha=vrt.read(vrt.count,out_dtype='float32')
        out=np.ma.filled(value,np.nan).astype('float32')
        cov=np.clip(alpha/255.,0,1).astype('float32')
        out[cov<=0]=np.nan
    return out,cov,g

for site,names in UAV_MOSAICS.items():
    for name in names:
        ref=DF.loc[(DF.site==site)&(DF.mosaic==name)]
        assert not ref.empty,(site,name,'no reference cells')
        native,coverage,g=warp_raw_teacher(name,site,100)
        print('100m source resampled:',name,'grid',g['height'],g['width'],flush=True)
        rr=ref.row.to_numpy(dtype=int);cc=ref.col.to_numpy(dtype=int)
        ok=(rr>=0)&(cc>=0)&(rr<g['height'])&(cc<g['width'])
        rr,cc=rr[ok],cc[ok]
        known=ref.mean_probability.to_numpy(float)[ok]
        vals=native[rr,cc];cv=coverage[rr,cc]
        good=(cv>=.90)&np.isfinite(vals)&np.isfinite(known)
        assert good.sum()>=8,(name,'insufficient covered reference cells',good.sum())
        candidates=[]
        for denom in SCALE_CANDIDATES:
            pred=vals[good]/denom
            mae=float(np.mean(abs(pred-known[good])))
            candidates.append((mae,denom,float(np.percentile(pred,99))))
        candidates.sort()
        mae,denom,p99=candidates[0]
        REFERENCE_DIAGNOSTICS.append(dict(site=site,mosaic=name,n=int(good.sum()),
            denominator=denom,MAE_fraction=mae,p99_prediction=p99,
            median_reference=float(np.median(known[good])),
            median_reconstructed=float(np.median(vals[good]/denom))))
        assert mae<.05 and p99<=1.02, (
            'Teacher calibration failed: inspect NoData and NB04 100m target construction',
            name,candidates)
        UAV_SCALE[name]=denom
        print('CALIBRATED',site,name,'denominator',denom,'100m MAE',round(mae,5))
CALIBRATION=pd.DataFrame(REFERENCE_DIAGNOSTICS)
display(CALIBRATION.round(4))
CALIBRATION.to_csv(OUT/'07_teacher_encoding_validation.csv',index=False)
assert len(UAV_SCALE)==5


## 08 · Visual UAV–Planet co-registration panels for all five mosaics

Generate a **common CRS/window footprint** from the source UAV raster. Compare UAV RGB, raw annual Planet RGB and UAV model probability side by side. The centre is chosen from an existing 100 m reference cell in the corresponding mosaic. This checks that roads, clearings and canopy boundaries are in consistent locations. **Different RGB stretches** are display-only; source rasters remain unchanged.


In [ ]:
# 08 — Side-by-side common-window comparison, display only.
from rasterio.windows import from_bounds as bounds_window
from pyproj import Transformer
from rasterio.warp import reproject,Resampling

PANEL_M=350; PANEL_N=420
def preview_on_grid(filepath,world_bounds,crs,bands):
    out=np.zeros((len(bands),PANEL_N,PANEL_N),dtype='float32')
    xmin,ymin,xmax,ymax=world_bounds
    transform=Affine((xmax-xmin)/PANEL_N,0,xmin,0,-(ymax-ymin)/PANEL_N,ymax)
    with rasterio.open(filepath) as ds:
        assert ds.crs
        for j,b in enumerate(bands):
            reproject(rasterio.band(ds,b),out[j],src_transform=ds.transform,
                src_crs=ds.crs,src_nodata=ds.nodata,
                dst_transform=transform,dst_crs=crs,dst_nodata=-9999.,
                resampling=Resampling.bilinear)
    out[out==-9999]=np.nan
    return out
def show_rgb(cube):
    valid=np.isfinite(cube).all(axis=0)&(cube>0).all(axis=0)
    img=np.ones((*valid.shape,3),dtype='float32')
    if valid.sum()>5:
        lo=np.percentile(cube[:,valid],2,axis=1);hi=np.percentile(cube[:,valid],98,axis=1)
        img[valid]=np.clip(((cube-lo[:,None,None])/(hi[:,None,None]-lo[:,None,None]+1e-6)).transpose(1,2,0)[valid],0,1)
    return img
fig,axes=plt.subplots(5,3,figsize=(13,20),layout='constrained')
for i,(site,names) in enumerate(UAV_MOSAICS.items()):
    pass
idx=0
for site,names in UAV_MOSAICS.items():
    for name in names:
        ref=DF.loc[(DF.site==site)&(DF.mosaic==name)].sort_values('mean_probability')
        pick=ref.iloc[len(ref)//2]
        g=satellite_grid(site,100)
        cx,cy=g['transform']*(float(pick.col)+.5,float(pick.row)+.5)
        half=PANEL_M/2
        bounds=(cx-half,cy-half,cx+half,cy+half)
        with rasterio.open(MOSAIC_PATH[name]) as ds:
            if ds.count<3:raise ValueError(('Orthomosaic has fewer than 3 bands',name))
        uav_rgb=preview_on_grid(MOSAIC_PATH[name],bounds,g['crs'],[1,2,3])
        # The Planet archive is B,G,R,N; raw median R,G,B = 15,14,13.
        planet_rgb=preview_on_grid(planet_file(site,'Annual'),bounds,g['crs'],[15,14,13])
        teacher=preview_on_grid(teacher_file(name),bounds,g['crs'],[1])[0]/UAV_SCALE[name]
        axes[idx,0].imshow(show_rgb(uav_rgb))
        axes[idx,1].imshow(show_rgb(planet_rgb))
        axes[idx,2].imshow(np.ma.masked_invalid(teacher),cmap='magma',vmin=0,vmax=1)
        axes[idx,0].set_title(f'{site} {Path(name).stem}: UAV RGB')
        axes[idx,1].set_title(f'Planet {PLANET_YEAR[site]} annual RGB')
        axes[idx,2].set_title('UAV teacher Musanga probability')
        for ax in axes[idx]:ax.axis('off')
        idx+=1
plt.savefig(FIG/'08_uav_planet_teacher_alignment.png',dpi=155,bbox_inches='tight')
plt.show()
print('Compare road lines, canopy edges and clearings; report offsets before training.')


## 09 · Build a *new* 50 m continuous UAV target dataset on the canonical S2 grid

No upsampling of 100 m labels. For each source orthomosaic separately, aggregate the original UAV probability raster to the exact 50 m target lattice. Treat overlapping MOS mosaics as distinct **groups** and flag cell duplicates; this permits grouped validation later. Enforce a minimum UAV coverage of 90% in each target cell, plus the original ROI.shp footprint covering ≥90% of the cell. Planet annual and embeddings must have valid center pixels; full-patch coverage is assessed separately during patch extraction.

This step caches only lightweight 50 m label tables and geometry indexes, **not** 64-band patches; NN input patches will be extracted from native imagery on demand. No model is fitted yet.


In [ ]:
# 09 — Build continuous 50m UAV labels (per-mosaic), enforce NB04B-style ROI/coverage rules.
import geopandas as gpd
from rasterio.features import rasterize
ROI_SOURCE=Path(uav_manifest['roi'])
assert ROI_SOURCE.is_file()
ROI_GDF=gpd.read_file(ROI_SOURCE)
assert ROI_GDF.crs and not ROI_GDF.empty
TARGET_SCALE=50
MIN_UAV_COVERAGE=.90
MIN_ROI_FRAC=.90
SUPERSAMPLE=4
TARGETS=[]
for site,names in UAV_MOSAICS.items():
    g=satellite_grid(site,TARGET_SCALE)
    polys=ROI_GDF.to_crs(g['crs'])
    fine_transform=g['transform']*Affine.scale(1/SUPERSAMPLE,1/SUPERSAMPLE)
    fine=rasterize(((geom,1) for geom in polys.geometry if geom is not None and not geom.is_empty),
        out_shape=(g['height']*SUPERSAMPLE,g['width']*SUPERSAMPLE),
        transform=fine_transform,fill=0,dtype='uint8')
    roif=fine.reshape(g['height'],SUPERSAMPLE,g['width'],SUPERSAMPLE).mean(axis=(1,3))
    del fine
    for name in names:
        values,coverage,_=warp_raw_teacher(name,site,TARGET_SCALE)
        probs=values/UAV_SCALE[name]
        ok=(coverage>=MIN_UAV_COVERAGE)&(roif>=MIN_ROI_FRAC)&np.isfinite(probs)&(probs>=0)&(probs<=1)
        yy,xx=np.where(ok)
        for row,col in zip(yy,xx):
            x,y=g['transform']*(int(col)+.5,int(row)+.5)
            TARGETS.append(dict(site=site,mosaic=name,group=Path(name).stem,
                year_uav=EMBEDDING_YEAR[site],year_planet=PLANET_YEAR[site],
                scale_m=TARGET_SCALE,row=int(row),col=int(col),
                x=float(x),y=float(y),crs=str(g['crs']),
                mean_probability=float(probs[row,col]),
                uav_coverage=float(coverage[row,col]),
                roi_fraction=float(roif[row,col])))
        print('50m supervised cells',site,name,len(yy),flush=True)
assert TARGETS, 'No 50m cells: inspect calibration / ROI / CRS'
T50=pd.DataFrame(TARGETS)
assert not T50.duplicated(['site','mosaic','row','col']).any()
T50.to_parquet(OUT/'09_uav_targets_50m.parquet',index=False)
display(T50.groupby(['site','group']).agg(cells=('mean_probability','size'),
    mean_fraction=('mean_probability','mean'),
    max_fraction=('mean_probability','max'),
    min_uav_coverage=('uav_coverage','min')).round(3))
print('Saved verified 50m UAV-derived labels:',OUT/'09_uav_targets_50m.parquet')


## 10 · Extract per-cell spatial features from **both** Planet and embeddings

Use the *same* 50 m target centers as Section 09. For each:
- Planet raw annual median B/G/R/N **local spectral means + contrast**, and sampled 9-band annual NIR texture.
- 64-band embedding **central values**, context means, context SDs, and mean absolute difference from the center.
- Include pixel-validity fractions for each native patch and reject edge/padding contamination below 80%.

Start by processing a **bounded pilot** of 300 target cells per forest to avoid excessive Google Drive random reads. This exploratory dataset is **not the final ML feature matrix**. Do not train or interpret prediction skill from it. Once this works, expand cell coverage and enforce strict holdout on any learned representation.


In [ ]:
# 10 — Efficient first spatial-feature extraction (small stratified pilot).
FEATURE_SAMPLE_PER_SITE=300
PLANET_PATCH=21   # ~100m context at 4.77m native scale
EMB_PATCH=11      # 110m context at 10m native scale
MIN_PATCH_VALID=.80
rng=np.random.default_rng(RANDOM_SEED)
SAMPLE50=[]
for site,local in T50.groupby('site',sort=False):
    # Coverage of Musanga range: bin by quantiles first then sample within bins.
    local=local.copy()
    local['stratum']=pd.qcut(local.mean_probability.rank(method='first'),
        q=min(5,len(local)),labels=False)
    parts=[]
    cap=max(1,int(np.ceil(FEATURE_SAMPLE_PER_SITE/local.stratum.nunique())))
    for _,sub in local.groupby('stratum'):
        parts.append(sub.sample(n=min(cap,len(sub)),random_state=RANDOM_SEED))
    take=pd.concat(parts).drop_duplicates(['mosaic','row','col']).head(FEATURE_SAMPLE_PER_SITE)
    SAMPLE50.append(take.drop(columns='stratum'))
PILOT50=pd.concat(SAMPLE50,ignore_index=True)
spatial_rows=[]
for site,local in PILOT50.groupby('site',sort=False):
    with rasterio.open(planet_file(site,'Annual')) as pds,\
         rasterio.open(planet_file(site,'Texture')) as tds,\
         rasterio.open(embed_file(site)) as eds:
        for rec in local.itertuples(index=False):
            def cube(ds,bands,size):
                px,py=Transformer.from_crs(rec.crs,ds.crs,always_xy=True).transform(rec.x,rec.y)
                cc,rr=~ds.transform*(px,py)
                win=Window(int(np.floor(cc-size/2)),int(np.floor(rr-size/2)),size,size)
                return ds.read(bands,window=win,boundless=True,fill_value=0).astype('float32')
            p=cube(pds,[13,14,15,16],PLANET_PATCH)
            t=cube(tds,list(range(1,10)),PLANET_PATCH)
            e=cube(eds,list(range(1,65)),EMB_PATCH)
            pm=np.isfinite(p).all(axis=0)&(p>0).all(axis=0)
            em=np.isfinite(e).all(axis=0)&np.any(e!=0,axis=0)
            if pm.mean()<MIN_PATCH_VALID or em.mean()<MIN_PATCH_VALID:continue
            px=p[:,pm];ex=e[:,em]
            center=e[:,EMB_PATCH//2,EMB_PATCH//2]
            row=dict(site=site,mosaic=rec.mosaic,group=rec.group,
                row=int(rec.row),col=int(rec.col),target_fraction=rec.mean_probability,
                planet_valid=float(pm.mean()),embedding_valid=float(em.mean()))
            for i,band in enumerate(['B','G','R','N']):
                row[f'PLANET_mean_{band}']=float(px[i].mean())
                row[f'PLANET_std_{band}']=float(px[i].std())
            for i,b in enumerate(('NIR_CV_r1','NIR_CV_r3','NIR_CV_r7','NIR_highpass')):
                v=t[i,np.isfinite(t[i])&(t[i]>=0)]
                row[f'PLANET_texture_{b}']=float(np.median(v)) if v.size else np.nan
            emean=ex.mean(axis=1);esd=ex.std(axis=1)
            for i in range(64):
                row[f'EMB_center_{i+1:02d}']=float(center[i])
                row[f'EMB_mean_{i+1:02d}']=float(emean[i])
                row[f'EMB_sd_{i+1:02d}']=float(esd[i])
            spatial_rows.append(row)
    print('Processed site',site,'examples collected',len([x for x in spatial_rows if x['site']==site]),flush=True)
F50=pd.DataFrame(spatial_rows)
assert len(F50)>0,'No valid matched Planet and embedding patches.'
F50.to_parquet(OUT/'10_exploratory_spatial_features_50m.parquet',index=False)
display(F50.groupby('site').agg(n=('target_fraction','size'),
    mean_fraction=('target_fraction','mean'),
    planet_valid=('planet_valid','mean'),
    embedding_valid=('embedding_valid','mean')).round(3))
print('Saved exploratory features:',OUT/'10_exploratory_spatial_features_50m.parquet')
print('NEXT: inspect co-registration figure and calibrated teacher targets; train only after QA.')
